# Bilayer Graphene Potential Energy Surface (XC-Mix Demo)

This notebook demonstrates a **PES scan** for AB-stacked bilayer graphene using QEpy, and compares how different exchange-correlation choices change the interlayer binding curve and how we can use QEpy to use our own set of custom functionals to run DFT calculations.

We scan the interlayer distance $d$ and evaluate total energies for:

- `PBE`
- `RVV10`
- `0.75*PBE + 0.25*RVV10`
- `0.50*PBE + 0.50*RVV10`

## Why this example is useful

- **Compact 2D unit cell** (4 atoms, γ-centered k-mesh) → moderate computational cost
- **Clear physics** → semilocal PBE underbinds π-stacked layers; nonlocal-vdW corrects this
- **XC-mix sweep** → shows how the RVV10 weight shifts the equilibrium distance and binding depth
- **Custom Functionals** → We use a variety of combinations of functionals to create a set of custom functionals

## Notebook outputs

- `bilayer_graphene_pes.pdf` with relative-energy plots
- per-XC equilibrium distance estimate (`d_eq`) and minimum energy (`E_min`) printed in the output

In [ ]:
# --- optional pip installs ---
!pip install qepy f90wrap==0.2.16
!pip install dftpy
!pip install matplotlib

## 2) Imports and helpers

Import NumPy, ASE atom utilities, and QEpy PES helpers:
- `Driver.compute_pes` for scan orchestration
- `scf_xc_mix` for mixed-XC SCF evaluations
- `make_label` for readable XC labels in logs and plots

In [ ]:
import numpy as np
from ase.build import graphene
from qepy.driver import Driver

## 3) Build and update bilayer geometry

Define helper functions to:
- create an AB-stacked bilayer graphene cell
- update only the interlayer separation $d$ during the scan

In [ ]:
def make_bilayer_graphene_AB(a=2.46, vacuum=20.0, d=3.35):
    """AB-stacked bilayer graphene, primitive cell (4 atoms); compatible with older ASE versions."""
    g1 = graphene(a=a, vacuum=vacuum)
    g1.center(axis=2)

    g2 = g1.copy()

    cell = g1.get_cell()
    shift_xy = (cell[0] + cell[1]) / 3.0

    g2.positions += shift_xy
    g2.positions[:, 2] += d

    bilayer = g1 + g2
    bilayer.set_cell(cell)
    bilayer.set_pbc([True, True, False])
    bilayer.center(axis=2)

    return bilayer


def set_bilayer_distance(atoms, d):
    """Return a copy with interlayer distance d (Å); first half of atoms are layer1, second half layer2."""
    atoms = atoms.copy()
    n = len(atoms)
    n2 = n // 2

    z1 = np.mean(atoms.positions[:n2, 2])
    z2 = np.mean(atoms.positions[n2:, 2])
    zmid = 0.5 * (z1 + z2)

    new_z1 = zmid - d / 2.0
    new_z2 = zmid + d / 2.0

    atoms.positions[:n2, 2] += (new_z1 - z1)
    atoms.positions[n2:, 2] += (new_z2 - z2)
    return atoms

## 4) Base QE options (`qe_options`)

Build the starting bilayer geometry and create a reusable QE input dictionary:
- SCF control settings
- plane-wave/system settings with smearing for semimetals
- electronic convergence settings
- atomic species, cell, k-point sampling

Only atomic positions are changed during the PES scan.

In [ ]:
# Base QE input options for bilayer graphene (d=3.35 A starting geometry, a=2.46 A, vacuum=20 A).
# We keep this dict explicit so learners can see exactly what is sent to QE.
qe_options = {
    "&control": {
        "calculation": "'scf'",
        "pseudo_dir": "'./data/'",
        "tprnfor": True,
        "tstress": True,
        "disk_io": "'none'",
        "verbosity": "'low'",
        "restart_mode": "'from_scratch'",
    },
    "&system": {
        "ibrav": 0,
        "ecutwfc": 60,
        "nat": 4,
        "ntyp": 1,
        "occupations": "'smearing'",
        "smearing": "'mv'",
        "degauss": 0.02,
        "nosym": True,
        "noinv": True,
    },
    "&electrons": {
        "conv_thr": 1e-9,
        "electron_maxstep": 200,
        "mixing_beta": 0.3,
    },
    "atomic_species": ["C 12.011 C_ONCV_PBE-1.2.upf"],
    "atomic_positions angstrom": [
        # Initial geometry at d=3.35 A (updated during the scan below).
        "C 0.0000000000 0.0000000000 18.3250000000",
        "C 1.2300000000 0.7101408311 18.3250000000",
        "C 0.4100000000 0.7101408311 21.6750000000",
        "C 1.6400000000 1.4202816622 21.6750000000",
    ],
    "cell_parameters angstrom": [
        "2.4600000000 0.0000000000 0.0000000000",
        "-1.2300000000 2.1304224933 0.0000000000",
        "0.0000000000 0.0000000000 40.0000000000",
    ],
    "k_points automatic": ["12 12 1 0 0 0"],
}

## Mixed-XC potential evaluator (`mix_xc`)

`eval_xc_mix` evaluates a *linear combination* ("mix") of exchange-correlation functionals for the driver's current electron density, using DFTpy:

- Pulls the density from the QEpy `driver` and reshapes it onto DFTpy's real-space field grid via `data2field`.
- For each `(name, coeff)` pair in `xc_mix` (e.g. `{"PBE": 0.5, "RVV10": 0.5}`), evaluates that functional with DFTpy and accumulates `coeff * potential` / `coeff * energy`. Terms with `coeff == 0.0` are skipped.
- The factor of `2` converts DFTpy's Hartree convention to the Rydberg units QEpy's `set_external_potential` expects.

Returns `(v_total, E_total)`, which `compute_pes` (below) feeds into each SCF iteration via `driver.set_external_potential(...)`.

In [3]:
def eval_xc_mix(driver, xc_mix):
    rho = driver.get_density()
    field = driver.data2field(rho)
    v_total = 0.0
    E_total = 0.0
    for name, coeff in xc_mix.items():
        if coeff == 0.0:
            continue
        xc = XC(name)
        func = xc(field)
        v_total = v_total + coeff * driver.field2data(func.potential) * 2
        E_total = E_total + coeff * func.energy * 2
    return v_total, E_tota

## PES scan driver (`compute_pes`)

`compute_pes` orchestrates the full scan: for every geometry in `scan_values`, it runs an SCF loop with a custom (possibly mixed) XC functional for each entry in `xc_list`, and collects the converged total energies.

Steps:
1. Build one label per XC mix in `xc_list` (e.g. `"0.5*PBE + 0.5*RVV10"`) from each mix's nonzero coefficients — these labels key the `energies` dict and name the per-point log files.
2. For each scan point:
   - Call `update_geometry(qe_options, val)` to get a fresh `qe_options` dict with the geometry updated for this point.
   - For each XC mix, create a QEpy `Driver` and run a manual SCF loop: evaluate the mixed XC potential/energy with `mix_xc`, feed it in via `set_external_potential`, then `diagonalize()` + `mix()` each iteration until `check_convergence()` or `maxiter` is reached.
   - Record the converged energy in `energies[label][i]` and stop the driver.
3. Return the scan grid and the `energies` dict (one energy array per XC label).

In [1]:
def compute_pes(scan_values, update_geometry, qe_options,
                 xc_list, scan_label="x", maxiter=80, log_prefix="pes"):
    scan_values = np.asarray(scan_values, dtype=float)
    n_points = len(scan_values)

    labels = []
    for xc in xc_list:
        parts = [name if c == 1.0 else f"{c:.2g}*{name}"
                 for name, c in xc.items() if c != 0.0]
        labels.append(" + ".join(parts) if parts else "XC")

    energies = {lab: np.zeros(n_points) for lab in labels}

    print(f"PES scan: {n_points} points along {scan_label}")
    print("=" * 60)
    for i, val in enumerate(scan_values):
        print(f"\n[{i + 1}/{n_points}] {scan_label} = {val:.6f}")
        opts = update_geometry(qe_options, val)

        for xc, lab in zip(xc_list, labels):
            logfile = f"{log_prefix}_{lab.replace(' ', '_')}_{val:.2f}.out"
            driver = Driver(qe_options=opts, iterative=True, logfile=logfile)
            for _ in range(maxiter):
                extpot, ex = mix_xc(driver, xc)
                driver.set_external_potential(potential=extpot, extene=ex, exttype="xc")
                driver.diagonalize()
                driver.mix()
                if driver.check_convergence():
                    break
            energies[lab][i] = driver.get_energy()
            driver.stop()
            print(f"  {lab}: E = {energies[lab][i]:.10f}")

    print("\n" + "=" * 60 + "\nPES scan complete.")
    return energies

## 5) XC definitions and geometry update

Set the XC mixes to compare and define `update_bilayer(qe_options, d)`, which returns updated `qe_options` for each interlayer distance $d$. `compute_pes` uses this together with `qe_options` and `xc_list` to run mixed-XC SCF at every grid point.

In [ ]:
# Reference atoms object used only to generate updated coordinates.
bilayer = make_bilayer_graphene_AB(a=2.46, vacuum=20.0, d=3.35)

# Compare semilocal, mixed, and pure nonlocal-vdW style curves.
# Keep list short to keep runtime manageable.
xc_list = [
    {"PBE": 1.0},
    {"RVV10": 1.0},
    {"PBE": 0.75, "RVV10": 0.25},
    {"PBE": 0.5, "RVV10": 0.5},
]

def update_bilayer(qe_opts, d):
    """Return qe_options with C positions replaced for interlayer distance d."""
    atoms_d = set_bilayer_distance(bilayer, d)
    qe_opts = dict(qe_opts)
    qe_opts["atomic_positions angstrom"] = [
        f"C {p[0]:.10f} {p[1]:.10f} {p[2]:.10f}"
        for p in atoms_d.positions
    ]
    return qe_opts

## 6) Run PES scan

Create an interlayer distance grid and call ``Driver.compute_pes(grid, update_bilayer, qe_options, xc_list, ...)``.

This returns:
- scanned distances (`d_grid`)
- an `energies` dict mapping each XC label to its total-energy array

The next section plots those arrays in memory—no CSV file is required.

In [ ]:
# Distance grid for the PES scan.
d_grid = np.linspace(2.8, 5.0, 20)

# Loops over d_grid; for each d, updates geometry then runs scf_xc_mix for each xc in xc_list.
energies = compute_pes(
    d_grid,
    update_bilayer,
    qe_options,
    xc_list,
    scan_label="d(A)",
    maxiter=50,
    log_prefix="pes_graphene",
)

## 7) Plot PES and report minima

Plot relative energies $(E - E_{\min})$ in eV for each XC curve,
and print:
- equilibrium distance estimate (`d_eq`)
- minimum absolute energy (`E_min`)

In [ ]:
import matplotlib.pyplot as plt

# Convert relative energies to eV for easier visual comparison.
Ha_to_eV = 27.2114
markers = ["o", "s", "^", "D"]
colors = ["#0b3c8c", "#1b9e77", "#d95f02", "#7570b3"]

fig, ax = plt.subplots(figsize=(8, 5))

# Plot one PES curve per XC definition.
for idx, (label, E) in enumerate(energies.items()):
    E_rel = (E - E.min()) * Ha_to_eV
    ax.plot(
        d_grid,
        E_rel,
        marker=markers[idx % len(markers)],
        color=colors[idx % len(colors)],
        label=label,
        linewidth=2,
        markersize=7,
    )
    # Print equilibrium distance and absolute minimum energy for each curve.
    d_eq = d_grid[np.argmin(E)]
    print(f"{label}: d_eq = {d_eq:.3f} Å, E_min = {E.min():.10f} Ha")

ax.set_xlabel(r"Interlayer distance $d$ ($\mathrm{\AA}$)", fontsize=12)
ax.set_ylabel(r"$E - E_{\mathrm{min}}$ (eV)", fontsize=12)
ax.set_title("Bilayer graphene PES (XC-mix comparison)", fontsize=13)
ax.set_xlim(d_grid.min(), d_grid.max())
ax.set_ylim(-0.02, None)
ax.legend(fontsize=10, loc="best")
ax.grid(True, which="both", ls="--", alpha=0.4)
ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)

plt.tight_layout()
plt.savefig("bilayer_graphene_pes.pdf", bbox_inches="tight")
plt.show()